In [4]:
# Colab cell
!pip install -q sentence-transformers faiss-cpu transformers fastapi uvicorn[standard] gradio pytest

In [5]:
# Colab cell
import os, json, time
from typing import List, Dict
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
print("Imports ready")

Imports ready


In [6]:
# Colab cell
os.makedirs("data", exist_ok=True)

docs = [
    {"id":"doc_1","title":"Company Overview","text":"Our company specializes in AI-driven solutions for healthcare and finance. Founded in 2015, we have offices in Cape Town and Johannesburg."},
    {"id":"doc_2","title":"Product Info","text":"MedAI is a diagnostic support tool that helps doctors analyze patient data using machine learning models. It reduces diagnostic errors by 20%."},
    {"id":"doc_3","title":"Employee Policy","text":"Employees are entitled to 20 days of annual leave. Remote work is supported with approval from management. All staff must complete cybersecurity training annually."}
]

qa_pairs = [
    {"q":"Tell me about MedAI","a":"MedAI is a diagnostic support tool that reduces diagnostic errors by 20%"},
    {"q":"How many annual leave days do employees get?","a":"20 days"},
    {"q":"Can employees work remotely?","a":"Yes, with management approval"},
    {"q":"What industries does the company specialize in?","a":"healthcare and finance"},
    {"q":"What training must staff complete?","a":"Annual cybersecurity training"}
]

with open("data/meta.json","w") as f:
    json.dump(docs, f, indent=2)
with open("data/qa.json","w") as f:
    json.dump(qa_pairs, f, indent=2)

print("Saved synthetic dataset to data/")

Saved synthetic dataset to data/


In [7]:
# Colab cell
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [d["text"] for d in docs]
embeddings = embed_model.encode(texts, convert_to_numpy=True)

# normalize for cosine similarity
faiss.normalize_L2(embeddings)
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

# persist
faiss.write_index(index, "data/faiss.index")
with open("data/meta.json","w") as f:
    json.dump(docs, f, indent=2)

print("FAISS index built and saved to data/faiss.index")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built and saved to data/faiss.index


In [8]:
# Colab cell
index = faiss.read_index("data/faiss.index")
with open("data/meta.json") as f:
    meta = json.load(f)

def retrieve(query: str, top_k: int = 1):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx == -1:
            continue
        results.append({"score": float(score), "doc": meta[idx]})
    return results

# quick test
print(retrieve("Tell me about MedAI"))


[{'score': 0.717105507850647, 'doc': {'id': 'doc_2', 'title': 'Product Info', 'text': 'MedAI is a diagnostic support tool that helps doctors analyze patient data using machine learning models. It reduces diagnostic errors by 20%.'}}]


In [9]:
# Colab cell
# Lightweight extractive summarizer (safe for low RAM)
def extractive_summary(text: str, max_chars: int = 140):
    if not text:
        return ""
    s = text.split(".")[0].strip()
    if len(s) > max_chars:
        return s[:max_chars].rsplit(" ",1)[0] + "..."
    return s + "."

# Optional transformer summarizer (comment/uncomment if you have RAM)
# summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
# def transformer_summary(text):
#     out = summarizer(text, max_length=60, min_length=10, do_sample=False)
#     return out[0]["summary_text"]

print("Summarizer ready (extractive).")


Summarizer ready (extractive).


In [10]:
# Colab cell
def ask(query: str):
    hits = retrieve(query, top_k=1)
    if not hits:
        return {"question": query, "answer": "Sorry, I don't have an answer for that yet."}
    raw = hits[0]["doc"]["text"]
    summary = extractive_summary(raw)
    return {"question": query, "answer": summary, "source": hits[0]["doc"]["title"], "score": hits[0]["score"]}

# Demo
print(ask("How many annual leave days do employees get?"))


{'question': 'How many annual leave days do employees get?', 'answer': 'Employees are entitled to 20 days of annual leave.', 'source': 'Employee Policy', 'score': 0.6096934676170349}


In [11]:
# Colab cell
import gradio as gr

def gradio_ask(q):
    return ask(q)["answer"]

demo = gr.Interface(fn=gradio_ask, inputs="text", outputs="text", title="AI Automation Engine Demo")
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0ccc29c01b3855735.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# Colab cell
# Simple inline tests
assert "MedAI" in ask("Tell me about MedAI")["answer"] or "diagnostic" in ask("Tell me about MedAI")["answer"]
assert "20" in ask("How many annual leave days do employees get?")["answer"]
print("Basic tests passed")


Basic tests passed
